# B2: RMSNorm

```
y = x / sqrt(mean(x^2) + eps) * weight        (mean over the 2048 hidden values of ONE token)
```

- Input and output shape: `(N, 2048)`: every token is normalised on its own.
- **LayerNorm** subtracts the mean and divides by the standard deviation, and adds a bias. **RMSNorm** only rescales by the root-mean-square: no mean, no bias, one reduction instead of two.
- The statistic is computed in **FP32** even for FP16 data. We show below why.
- TinyLlama has **45** of these: input and post-attention norm in each of 22 layers, plus a final norm.

Code: `tinyserve/models/norm.py`. Tests: `tests/test_b2_rmsnorm.py`. The reference is Hugging Face's `LlamaRMSNorm`.

In [ ]:
# Setup: clone or hard-reset the repo to GitHub, cd into it, put it on sys.path, install pytest.
import os, subprocess, sys

REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    # Colab may hold untracked files (e.g. results written by a notebook) that clash with a pull.
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
import importlib, json
import torch
import tinyserve.quiet, tinyserve.basics, tinyserve.models.norm, tinyserve.probe
for module in (tinyserve.quiet, tinyserve.basics, tinyserve.models.norm, tinyserve.probe):
    importlib.reload(module)  # picks up repo edits without restarting the kernel
from tinyserve.envinfo import collect_env
from tinyserve.models.norm import RMSNorm, rms_norm

print(collect_env())

## 1. By hand
`x = [3, 4]`: `mean(x^2) = (9 + 16) / 2 = 12.5`, `rms = sqrt(12.5) = 3.5355`, so `y = [3, 4] / 3.5355 = [0.8485, 1.1314]`. The output always has RMS 1 (before the weight scales it).

In [ ]:
small = torch.tensor([[3.0, 4.0]])
print("ours:", rms_norm(small, torch.ones(2), eps=0.0))
print("by hand:", [3 / 12.5 ** 0.5, 4 / 12.5 ** 0.5])

## 2. It does not subtract the mean
Add 5 to every value. LayerNorm removes the shift; RMSNorm does not, because the shift changes the RMS and the direction of the vector.

In [ ]:
generator = torch.Generator().manual_seed(0)
sample = torch.randn(1, 8, generator=generator)
layer_norm = torch.nn.LayerNorm(8, elementwise_affine=False)
ones = torch.ones(8)
print("x                  :", sample[0].numpy().round(2))
print("LayerNorm(x)       :", layer_norm(sample)[0].detach().numpy().round(2))
print("LayerNorm(x + 5)   :", layer_norm(sample + 5)[0].detach().numpy().round(2), "  <- same")
print("RMSNorm(x)         :", rms_norm(sample, ones, 1e-5)[0].numpy().round(2))
print("RMSNorm(x + 5)     :", rms_norm(sample + 5, ones, 1e-5)[0].numpy().round(2), "  <- different")

## 3. Why the statistic is computed in FP32
FP16 tops out at 65504. A value of 300 squares to 90000, which overflows to `inf`. In a naive all-FP16 version `mean = inf`, `rsqrt(inf) = 0`, and the output becomes **all zeros, with no error raised**. Our version squares in FP32.

In [ ]:
big = torch.full((1, 8), 300.0, dtype=torch.float16, device="cuda")
naive = big * torch.rsqrt(big.pow(2).mean(-1, keepdim=True) + 1e-5)  # everything in FP16
ours = rms_norm(big, torch.ones(8, dtype=torch.float16, device="cuda"), 1e-5)
print("x            :", big[0, :4].tolist())
print("naive FP16   :", naive[0, :4].tolist(), " (silently zero)")
print("ours (FP32 stats):", ours[0, :4].tolist())

## 4. Against the real model, on real activations
We run one prompt through the Hugging Face TinyLlama (FP16, GPU), record the input and output of all 45 `LlamaRMSNorm` modules with hooks, feed each real input to **our** `RMSNorm` (with the same weight), and compare. Because our operation order matches Hugging Face's, we expect the FP16 outputs to be **exactly** equal.

In [ ]:
from transformers.models.llama.modeling_llama import LlamaRMSNorm
from tinyserve.basics import load_model, encode
from tinyserve.probe import compare_rmsnorm

tokenizer, model = load_model(device="cuda", dtype=torch.float16)
prompt_ids = encode(tokenizer, "The capital of France is", "cuda")
rows = compare_rmsnorm(model, prompt_ids, LlamaRMSNorm, eps=model.config.rms_norm_eps)

print(f"norms compared: {len(rows)}   exactly equal: {sum(r['exactly_equal'] for r in rows)}   worst |diff|: {max(r['max_abs_diff'] for r in rows)}")
print()
print(f"{'module':<45} {'input RMS':>10} {'input max|x|':>13} {'max |diff|':>11}")
for row in rows:
    if any(key in row['name'] for key in ('layers.0.', 'layers.1.input', 'layers.10.input', 'layers.21.', 'model.norm')):
        print(f"{row['name']:<45} {row['input_rms']:10.3f} {row['input_max_abs']:13.1f} {row['max_abs_diff']:11.1e}")

Compare the `input max|x|` column with the `input RMS` column: how large is the largest hidden value relative to a typical one, and how close does it get to the FP16 limit? The next cell ranks the norms by that ratio.

In [ ]:
ratio_rows = sorted(rows, key=lambda r: -r["input_max_abs"] / r["input_rms"])[:3]
for row in ratio_rows:
    print(f"{row['name']:<45} max|x| = {row['input_max_abs']:8.1f}   RMS = {row['input_rms']:7.3f}   ratio {row['input_max_abs'] / row['input_rms']:6.1f}x")
print("FP16 maximum is 65504; squaring overflows above", round(65504 ** 0.5, 1))
print("largest |x| seen anywhere:", max(r['input_max_abs'] for r in rows))

## 5. Save results and run the tests

In [ ]:
from tinyserve.envinfo import save_result

saved_path = save_result("B2_rmsnorm", {
    "prompt": "The capital of France is",
    "dtype": "float16",
    "norms_compared": len(rows),
    "exactly_equal": sum(r["exactly_equal"] for r in rows),
    "worst_abs_diff": max(r["max_abs_diff"] for r in rows),
    "per_norm": rows,
})
print(saved_path)

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "tests/test_b2_rmsnorm.py", "-v"], capture_output=True, text=True)
print(result.stdout[-4000:], result.stderr[-1000:])

In [ ]:
# Print the full results file so it can be copied into the repo from the saved notebook.
print(open(saved_path).read())